# RunPod GPU
Each session: **2 Start → 3 Connect → 4 Prepare → work → 5 Delete → 6 Check**  ·  💻 local kernel (`.venv`)  ·  🟢 GPU kernel

## 0 · Setup (once per computer)
Mac: run the cell. Linux: in a terminal, `curl -sSL https://cli.runpod.net | sudo bash`.
Then in a terminal: `runpodctl doctor` and paste an [API key](https://console.runpod.io/user/settings) (read/write). Never put the key in a notebook.

In [ ]:
!brew install runpod/runpodctl/runpodctl && runpodctl version

## 1 · GPUs 💻 (optional)
Lists available GPUs and prices, cheapest first. 24 GB is enough here.

In [ ]:
import json, subprocess

gpus = json.loads(subprocess.run(["runpodctl", "gpu", "list"], capture_output=True, text=True).stdout)
for g in sorted(gpus, key=lambda g: g.get("securePricePerHr") or 1e9):
    if g.get("securePricePerHr") and g["gpuId"].startswith("NVIDIA") and "MIG" not in g["gpuId"]:
        print(f'${g["securePricePerHr"]:<5} {g["memoryInGb"]:>3} GB  {g["gpuId"]}')

## 2 · Start 💻
Starts the first available GPU in `GPUS` (1–3 min; charges start). Prints GPU, price, start time and the link. On error, run step 6.

In [ ]:
import json, pathlib, secrets, subprocess, time

GPUS = ["NVIDIA RTX A5000", "NVIDIA A40", "NVIDIA GeForce RTX 4090", "NVIDIA L40S"]  # first available
rp = lambda *a: subprocess.run(["runpodctl", *a], capture_output=True, text=True)
token, t0 = secrets.token_urlsafe(12), time.time()
for gpu in GPUS:
    r = rp("pod", "create", "--name", "learn-gpu", "--template-id", "runpod-torch-v280", "--gpu-id", gpu,
           "--min-cuda-version", "12.8", "--env", json.dumps({"JUPYTER_PASSWORD": token}), "--wait")
    if r.returncode == 0 or '"id"' in r.stderr: break
    print(gpu, "unavailable")
if r.returncode: raise RuntimeError(r.stderr)
pod_id = json.loads(r.stdout)["id"]
price = next(g["securePricePerHr"] for g in json.loads(rp("gpu", "list", "--include-unavailable").stdout) if g["gpuId"] == gpu)
pathlib.Path("~/.runpod/session.json").expanduser().write_text(json.dumps({"t0": t0, "price": price}))
print(f"{gpu} · ${price}/hr · ready in {time.time() - t0:.0f} s")
print(f"https://{pod_id}-8888.proxy.runpod.net/?token={token}")

## 3 · Connect (clicks)
Top-right **Select Kernel** → **Select Another Kernel…** → **Existing Jupyter Server…** → paste the link → Enter → Enter → **Python 3 (ipykernel)**.
Do the same in any notebook you want on the GPU (e.g. `PreTraining.ipynb`).

## 4 · Prepare 🟢
Copies the repo from GitHub (push first) to `/workspace`, installs `tiktoken`, makes GPU notebooks start there, auto-deletes the GPU after 2 h.

In [ ]:
import os, subprocess, time, torch
print("GPU:", torch.cuda.get_device_name(0))
!git clone -q https://github.com/faisalanjum/LLM-fine-tuning /tmp/repo && cp -r /tmp/repo/. /workspace/
!pip install -q --break-system-packages tiktoken requests
!mkdir -p ~/.ipython/profile_default/startup && echo 'import os; os.chdir("/workspace")' > ~/.ipython/profile_default/startup/00-workspace.py
os.chdir("/workspace")
timer = subprocess.Popen("sleep 7200; runpodctl remove pod $RUNPOD_POD_ID", shell=True,
                         start_new_session=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(1)
print("Auto-delete armed: 2 h" if timer.poll() is None else "Auto-delete FAILED")

## Example 🟢 (optional)
Moves data and a model to the GPU. Your notebooks already do this via `device`.

In [ ]:
import time, torch

device = torch.device("cuda")
x = torch.randn(8000, 8000)                         # data on CPU
t = time.time(); x @ x; print(f"CPU: {time.time() - t:.2f} s")
x = x.to(device); x @ x; torch.cuda.synchronize()   # move to GPU (+ warm-up)
t = time.time(); x @ x; torch.cuda.synchronize(); print(f"GPU: {time.time() - t:.3f} s")
model = torch.nn.Linear(8000, 10).to(device)        # move a model to GPU
print(model(x).shape, next(model.parameters()).device)

## 5 · Delete 🟢
Deletes the GPU; charges stop. The kernel then dies — expected.

In [ ]:
import time; t = time.time()
!runpodctl remove pod $RUNPOD_POD_ID 2>&1 | grep -v "config file"
print(f"Deleted in {time.time() - t:.1f} s")

## 6 · Check 💻
**Select Kernel** → **Select Another Kernel…** → **Python Environments…** → `.venv`. Deletes any leftover, shows `[]` (nothing running), session time and cost.

In [ ]:
import json, pathlib, subprocess, time

rp = lambda *a: subprocess.run(["runpodctl", *a], capture_output=True, text=True).stdout
for p in json.loads(rp("pod", "list", "--all")):
    if p.get("name") == "learn-gpu": rp("pod", "delete", p["id"]); print("Deleted leftover", p["id"])
print("Running:", json.loads(rp("pod", "list", "--all")))
s = json.loads(pathlib.Path("~/.runpod/session.json").expanduser().read_text())
mins = (time.time() - s["t0"]) / 60
print(f"Session: {mins:.0f} min · ≈ ${s['price'] * mins / 60:.2f}")